### Convert annotations to the format used by GLiNER for finetuning.

Define the imports

In [10]:
import json
import re

Define paths to the annotation files

In [11]:
with open("data/val_split.json", "r") as file:
    docs_val = json.load(file)
val_pmid = docs_val.keys()
print(len(val_pmid), val_pmid)

40 dict_keys(['39166107', '40665707', '40042065', '28716445', '37849234', '39392298', '34205336', '33963227', '38480804', '37802772', '37403161', '33963281', '38151560', '37676767', '25869281', '33777957', '38850348', '28556833', '33195226', '30309367', '39019369', '37422636', '37451747', '39998261', '36253847', '37212075', '33327540', '39579960', '34480631', '36804535', '31662903', '31530799', '34290352', '35674870', '32438623', '37693595', '34560239', '36731694', '38613087', '37071196'])


In [12]:
PATH_GOLD_TRAIN = "./data/train_gold.json"
PATH_SILVER_TRAIN = "./data/train_silver.json"
PATH_SILVER_TRAIN_2025 = "./data/train_silver_2025.json"
# PATH_BRONZE_TRAIN = "../../Annotations/Train/bronze_quality/json_format/train_bronze.json"
PATH_DEV = "dev.json"

Define output paths

In [8]:
# #PATH_OUTPUT_PLATINUM_TRAIN = "../Train/NER/data/train_platinum.json"
PATH_OUTPUT_GOLD_TRAIN = "./data/train_gold.json"
PATH_OUTPUT_SILVER_TRAIN = "./data/train_silver.json"
PATH_OUTPUT_SILVER_TRAIN_2025 = "./data/train_silver_2025.json"
# PATH_OUTPUT_BRONZE_TRAIN = "../Train/NER/data/train_bronze.json"
PATH_OUTPUT_DEV = "dev_processed.json"

Load the input files into dictionary variables

In [18]:
with open(PATH_GOLD_TRAIN, 'r', encoding='utf-8') as file:
	train_gold = json.load(file)

with open(PATH_SILVER_TRAIN, 'r', encoding='utf-8') as file:
	train_silver = json.load(file)

with open(PATH_SILVER_TRAIN_2025, 'r', encoding='utf-8') as file:
	train_silver_2025 = json.load(file)
	
# with open(PATH_BRONZE_TRAIN, 'r', encoding='utf-8') as file:
# 	train_bronze = json.load(file)

with open(PATH_DEV, 'r', encoding='utf-8') as file:
	dev = json.load(file)

train = {**train_gold, **train_silver, **train_silver_2025}

val = {k: v for k, v in dev.items() if k in val_pmid}
test = {k: v for k, v in dev.items() if k not in val_pmid}

In [25]:
with open("./data/raw_test.json", "w") as file:
    json.dump(test, file, indent=2)

Define the function to parse the annotations to the GLiNER finetuning format

In [19]:
def tokenize_text_with_positions(text):
    # Split text into tokens, preserving punctuation (except for hyphens and underscores)
    tokens = []
    token_spans = []  # list of (start_char_index, end_char_index) for each token
    pattern = re.compile(r"\w+|[.,!?;:\'\"()\[\]{}<>]|[\s]+|\S")
    for match in pattern.finditer(text):
        token = match.group()
        if token.isspace():
            continue  # Skip whitespace tokens
        start_pos = match.start()
        if re.match(r"\w+-\w+", token) or re.match(r"\w+_\w+", token):
            # Keep hyphenated or underscored words intact
            tokens.append(token)
            token_spans.append((start_pos, match.end()))
        else:
            # Split contractions (e.g., "don't" -> "don", "'", "t")
            contraction_match = re.match(r"(\w+)(')(\w+)", token)
            if contraction_match:
                groups = contraction_match.groups()
                for group in groups:
                    end_pos = start_pos + len(group)
                    tokens.append(group)
                    token_spans.append((start_pos, end_pos))
                    start_pos = end_pos
            else:
                tokens.append(token)
                token_spans.append((start_pos, match.end()))
    return tokens, token_spans

def process_annotations(data):
    output_data = []

    for doc_id, doc_data in data.items():
        overall_tokenized_text = []
        overall_ner = []
        overall_spans = []
        token_offset = 0

        fields = ["title", "abstract"]

        for field in fields:
            text = doc_data["metadata"].get(field, "")
            tokens, token_spans = tokenize_text_with_positions(text)

            # Collect entities for this field
            field_entities = []
            for entity in doc_data.get("entities", []):
                mention_location = entity.get("location", "")
                if mention_location == field:
                    field_entities.append(entity)

            # Map entities from character indices to token indices
            for entity in field_entities:
                entity_start_char = entity["start_idx"]
                entity_end_char = entity["end_idx"] + 1  # Adjusting end index to be exclusive
                entity_label = entity["label"]

                entity_start_token_index = None
                entity_end_token_index = None

                for i, (token_start_char, token_end_char) in enumerate(token_spans):
                    if token_end_char <= entity_start_char:
                        continue  # Token is before the entity
                    if token_start_char >= entity_end_char:
                        break  # Token is after the entity
                    # Token overlaps with entity
                    if entity_start_token_index is None:
                        entity_start_token_index = i
                    entity_end_token_index = i  # Update to the last overlapping token

                if entity_start_token_index is not None and entity_end_token_index is not None:
                    overall_ner.append([
                        entity_start_token_index + token_offset,
                        entity_end_token_index + token_offset,
                        entity_label.lower()
                    ])
                else:
                    print(f"Warning: Could not find tokens for entity in doc {doc_id}, field {field}")

            # Append tokens to the overall tokenized text
            overall_tokenized_text.extend(tokens)
            overall_spans.extend(token_spans)
            token_offset += len(tokens)

        # Sort the word positions by the start index
        overall_ner.sort(key=lambda x: x[0])

        # Create the output dictionary for this document
        output_doc = {
            "pmid": doc_id,
            "tokenized_text": overall_tokenized_text,
            "ner": overall_ner,
            "token_indexes": overall_spans
        }

        output_data.append(output_doc)

    return output_data

In [30]:
def chunking_gliner(entry, size=384, overlap=20):
    chunks = []

    tokens = entry["tokenized_text"]
    ner = entry["ner"]

    step = size - overlap

    for i in range(0, len(tokens), step):
        token_start = i
        token_end = min(i + size, len(tokens))

        chunk_tokens = tokens[token_start:token_end]

        chunk_ner = []

        for ent_start, ent_end, label in ner:
            ent_start = int(ent_start)
            ent_end = int(ent_end)

            # Mantener solo entidades completamente dentro del chunk
            if ent_start >= token_start and ent_end < token_end:
                new_start = ent_start - token_start
                new_end = ent_end - token_start

                chunk_ner.append([
                    new_start,
                    new_end,
                    label
                ])

        chunks.append({
            "tokenized_text": chunk_tokens,
            "ner": chunk_ner
        })

        if token_end >= len(tokens):
            break

    return chunks
def chunk_data(data):
    def_data = []
    for entry in data:
        
        fragments = chunking_gliner(entry)
        def_data.extend(fragments)
    return def_data

In [31]:
processed_train = process_annotations(train)
processed_dev = process_annotations(val)
chunked_val = chunk_data(processed_dev)
processed_test = process_annotations(test)

In [32]:
with open("processed_val_chunked.json", "w") as file:
    json.dump(chunked_val, file, indent = 2)

In [30]:
for (s, e, label), entry in zip(sample["ner"], dev["35766370"]["entities"]):
    text = ""
    tokens = sample["tokenized_text"][s:e+1]
    indexes = sample["token_indexes"][s:e+1]
    for i in range(len(tokens)):
        if i == 0: 
            text += tokens[i]
        else: 
            diferencia = indexes[i][0] - indexes[i-1][1]
            text += " " * diferencia
            text += tokens[i]
        
    print(s, e, label, sample["tokenized_text"][s:e+1], entry["text_span"])
    print(list(sample["token_indexes"][i] for i in range(s, e+1)), text)
    print()

3 5 ddf ['Gut', 'Microbiome', 'Dysbiosis'] Gut Microbiome Dysbiosis
[(26, 29), (30, 40), (41, 50)] Gut Microbiome Dysbiosis

7 9 ddf ['Intestinal', 'Barrier', 'Dysfunction'] Intestinal Barrier Dysfunction
[(55, 65), (66, 73), (74, 85)] Intestinal Barrier Dysfunction

11 14 human ['Prodromal', 'Alzheimer', 'Disease', 'Patients'] Prodromal Alzheimer Disease Patients
[(89, 98), (99, 108), (109, 116), (117, 125)] Prodromal Alzheimer Disease Patients

24 25 microbiome ['gut', 'microbiota'] gut microbiota
[(15, 18), (19, 29)] gut microbiota

35 36 human ['elderly', 'patients'] elderly patients
[(98, 105), (106, 114)] elderly patients

39 40 ddf ['normal', 'cognition'] normal cognition
[(127, 133), (134, 143)] normal cognition

42 42 ddf ['NC'] NC
[(145, 147)] NC

55 57 ddf ['subjective', 'cognitive', 'decline'] subjective cognitive decline
[(216, 226), (227, 236), (237, 244)] subjective cognitive decline

59 59 ddf ['SCD'] SCD
[(246, 249)] SCD

62 65 ddf ['amnestic', 'mild', 'cognitive', 'im

Dump to json

In [23]:
def dump_to_json(dict, output_file_path):
	with open(output_file_path, 'w', encoding='utf-8') as f:
		#json.dump(dict, f, indent=2)
		json.dump(dict, f)

dump_to_json(processed_train, "./data/processed_train.json")
dump_to_json(processed_dev, "./data/processed_val.json")
dump_to_json(processed_test, "./data/processed_test.json")

In [31]:
with open("predictions/18-05/mingxiaw20_base.json", "r") as file:
    preds = json.load(file)

In [55]:
sample = preds[0]
sample["token_indexes"] = index
print(sample)

{'pmid': '35766370', 'location': 'title', 'tokens': ['Orthopedic', 'Surgery', 'Causes', 'Gut', 'Microbiome', 'Dysbiosis', 'and', 'Intestinal', 'Barrier', 'Dysfunction', 'in', 'Prodromal', 'Alzheimer', 'Disease', 'Patients', ':', 'A', 'Prospective', 'Observational', 'Cohort', 'Study', '.'], 'predictions': ['O', 'O', 'O', 'B-ddf', 'I-ddf', 'I-ddf', 'O', 'B-ddf', 'I-ddf', 'I-ddf', 'O', 'B-human', 'I-human', 'I-human', 'I-human', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], 'token_indexes': [(0, 10), (11, 18), (19, 25), (26, 29), (30, 40), (41, 50), (51, 54), (55, 65), (66, 73), (74, 85), (86, 88), (89, 98), (99, 108), (109, 116), (117, 125), (125, 126), (127, 128), (129, 140), (141, 154), (155, 161), (162, 167), (167, 168)]}


In [51]:
todas= []
entidad = []

for i, label in enumerate(sample["predictions"]):
    if label == "O": 
        if entidad != []:
            todas.append(entidad)
            entidad = []
        continue
    letter, ent = label.split("-")
    if letter == "B":
        token = sample["tokens"][i]
        entidad.append((letter, ent, token))
    elif letter == "I":
        if entidad[len(entidad)-1][1] == ent:
            token = sample["tokens"][i]
            entidad.append((letter, ent, token))
        
        
print(todas)
print(len(sample["tokens"]))
    

[[('B', 'ddf', 'Gut'), ('I', 'ddf', 'Microbiome'), ('I', 'ddf', 'Dysbiosis')], [('B', 'ddf', 'Intestinal'), ('I', 'ddf', 'Barrier'), ('I', 'ddf', 'Dysfunction')], [('B', 'human', 'Prodromal'), ('I', 'human', 'Alzheimer'), ('I', 'human', 'Disease'), ('I', 'human', 'Patients')]]
22


In [ ]:
def juntar_entidades(predicciones):
    todas= []
entidad = []

for i, label in enumerate(sample["predictions"]):
    if label == "O": 
        if entidad != []:
            todas.append(entidad)
            entidad = []
        continue
    letter, ent = label.split("-")
    if letter == "B":
        entidad.append((letter, ent))
    elif letter == "I":
        if entidad[len(entidad)-1][1] == ent:
            entidad.append((letter, ent))
        
        
    print(i, label)
print(todas)

In [57]:
numeros  = list(range(22))
numeros[18:30]

[18, 19, 20, 21]

In [74]:
def chunking(tokens, token_spans, size = 256, overlap = 32):
    chunks = []
    step = size - overlap
    for i in range(0, len(tokens), step):
        token_start = i
        token_end = i+ size
        chunk = tokens[token_start:token_end]
        chunk_start = token_spans[i][0]
        chunk_end = token[len(chunk)-1][1]
        chunks.append({
            "chunk_start": chunk_start,
            "chunk_end":chunk_end, 
            "chunk_tokens": chunk
        })
        if token_end >= len(tokens):
            break
    return chunks

In [75]:

tokens = truth["tokenized_text"]
c = chunking(tokens)

In [76]:
c

[{'chunk_start': 0,
  'chunk_end': 256,
  'chunk_tokens': ['Orthopedic',
   'Surgery',
   'Causes',
   'Gut',
   'Microbiome',
   'Dysbiosis',
   'and',
   'Intestinal',
   'Barrier',
   'Dysfunction',
   'in',
   'Prodromal',
   'Alzheimer',
   'Disease',
   'Patients',
   ':',
   'A',
   'Prospective',
   'Observational',
   'Cohort',
   'Study',
   '.',
   'To',
   'investigate',
   'gut',
   'microbiota',
   'and',
   'intestinal',
   'barrier',
   'function',
   'changes',
   'after',
   'orthopedic',
   'surgery',
   'in',
   'elderly',
   'patients',
   'with',
   'either',
   'normal',
   'cognition',
   '(',
   'NC',
   ')',
   'or',
   'a',
   'prodromal',
   'Alzheimer',
   'disease',
   'phenotype',
   '(',
   'pAD',
   ')',
   'comprising',
   'either',
   'subjective',
   'cognitive',
   'decline',
   '(',
   'SCD',
   ')',
   'or',
   'amnestic',
   'mild',
   'cognitive',
   'impairment',
   '(',
   'aMCI',
   ')',
   '.',
   'Homeostatic',
   'disturbances',
   'induce